# 5G Network Intelligence — Target Patches and Past KPI Attention

Research question: can explicit target/covariate attention improve DLPRB MAE and WAPE? This is an independently implemented TimeXer-inspired adaptation, not a reproduction. See [research notes](../docs/literature_driven_training.md) for primary sources and limitations.

In [ ]:
# Review Run All does not train. Training belongs in a dedicated hosted worker.
RUN_TRAINING = False


## Temporal protocol and metrics
Context: 168 observed hours. Horizons: 1/6/24h. Fit/stop/score endpoints: 335/383/443 and 395/443/503. All original hours 504 onward are excluded. MAE uses physical DLPRB usage units; WAPE = sum absolute error / sum absolute observed load. These are reused development folds, not an untouched test.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display
PROJECT = Path.cwd()
if not (PROJECT/'configs').is_dir() and (PROJECT.parent/'configs').is_dir():
    PROJECT = PROJECT.parent
CONFIG = PROJECT/'configs/past_covariate_attention.yaml'
cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8')) if CONFIG.exists() else None
print(cfg if cfg else 'Open from the project checkout to inspect configuration.')


## Preprocessing and architecture
Fit global per-KPI medians/scales on the fit prefix only. Keep missing masks and phase4. Encode DLPRB into 12-hour patches; a global target token attends to three past-KPI tokens. Predict a correction to a causal seasonal baseline. The control receives constant covariate tokens with identical parameter capacity. No future covariates, beam identity embeddings or fitted beam summaries are inputs.

## Training controls
Seed 17, maximum 80 epochs, patience 12. Select checkpoints on stopping MAE only; score every declared horizon on the full population. Save cold states, scaler, best/last checkpoints, RNG and provenance in hosted storage. Implementation: `src/models/past_covariate_attention.py`; controller: `scripts/train_past_covariate_attention.py`. Review does not repeat fitting.

In [ ]:
if RUN_TRAINING:
    raise RuntimeError('Use a separately pinned hosted training worker; review never launches fitting.')
print('Review mode: no fitting.')


## Evaluation and debugging
Inspect both arms and all fold/horizon rows. Do not call label availability or parameter count an accuracy gain. Before accepting results, independently reconstruct metrics, verify preprocessing/cold states and replay forecasts with a fixed numerical tolerance.

In [ ]:
# Read completed reports only. Missing output means the experiment is not yet evaluated.
for case in ['no_covariates', 'past_covariates']:
    report = PROJECT/'reports'/f'past_covariate_attention_{case}_development.csv'
    if report.exists():
        frame = pd.read_csv(report)
        frame['WAPE (%)'] = frame['wape']*100
        print(case)
        display(frame)
    else:
        print(case, ': no completed report available')


## Decision and limitations
Initial screen: at least 2% mean MAE improvement against the fixed strong control without fold/horizon regression. A 5% material gain still needs uncertainty and robustness evidence. WAPE below 10% remains unproven. The prior Telecom data/budget study failed its quality gates; this new architectural hypothesis may also fail. No holdout evaluation or production promotion follows from this notebook.